# MODELO DE CLASIFICACIÓN DE IMAGENES PARA DETERMINAR NEUMONIA EN PACIENTES CON COVID 19 USANDO REDES NEURONALES CONVOLUCIONALES

## DATASET  [KAGGLE](https://www.kaggle.com/datasets/khoongweihao/covid19-xray-dataset-train-test-sets/data)

## AUTOR : CÉSAR MAYTA

## IMPORTAMOS LIBRERIAS

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CARGAMOS PATH DE IMAGENES

In [ ]:
import os
BASE_DIR = '/content/drive/MyDrive/NOTEBOOKS/MODULO6-DEEP-LEARNING/PROYECTO_FINAL/dataset'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VALIDATION_DIR = os.path.join(BASE_DIR,'validation')
TEST_DIR = os.path.join(BASE_DIR,'test')
print(TRAIN_DIR)
print(VALIDATION_DIR)
print(TEST_DIR)

## 1 -CREO LOS DIRECTORIOS PARA CADA CONJUNTO DE DATOS DE NORMAL Y PNEUMONIA

In [ ]:
train_normal_dir = os.path.join(TRAIN_DIR, 'NORMAL')
train_pneumonia_dir = os.path.join(TRAIN_DIR, 'PNEUMONIA')

validation_normal_dir = os.path.join(VALIDATION_DIR, 'NORMAL')
validation_pneumonia_dir = os.path.join(VALIDATION_DIR, 'PNEUMONIA')

test_normal_dir = os.path.join(TEST_DIR, 'NORMAL')
test_pneumonia_dir = os.path.join(TEST_DIR, 'PNEUMONIA')

print(train_normal_dir)
print(train_pneumonia_dir)
print(validation_normal_dir)
print(validation_pneumonia_dir)
print(test_normal_dir)
print(test_pneumonia_dir)

# MOSTRAMOS UNA IMAGEN

In [ ]:
from tensorflow.keras.preprocessing import image
img = image.load_img(f'{train_normal_dir}/IM-0001-0001.jpeg',target_size=(64,64))
plt.imshow(img)
plt.title('PLACA NORMAL')
plt.axis('off')
plt.show()

# Mostramos que cantidades de archivos hay en cada carpeta

In [ ]:
dataset_dir = [train_normal_dir,train_pneumonia_dir,validation_normal_dir,validation_pneumonia_dir,test_normal_dir,test_pneumonia_dir]
for img_dir in dataset_dir:
  print(f" en directorio :{img_dir} hay {len(os.listdir(img_dir))} imagenes")

# CREAMOS OBJETOS PARA CARGAR EL CONJUNTO DE ENTRENAMIENTO VALIDACIÓN Y PRUEBAS

In [6]:
import tensorflow as tf
print(tf.__version__)

2.18.0


# CREAMOS OBJETOS PARA CARGAR EL CONJUNTO DE ENTRENAMIENTO VALIDACIÓN Y PRUEBAS CON DATA AUGMENTATION

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_data = ImageDataGenerator(rescale=1. /255,
                                    rotation_range=40,
                                    width_shift_range=0.2,
                                    height_shift_range=0.2,
                                    shear_range=0.2,
                                    zoom_range=0.2,
                                    horizontal_flip=True)
val_data = ImageDataGenerator(rescale=1. /255)
test_data = ImageDataGenerator(rescale=1. /255)

## CARGAMOS LOS CONJUNTOS DE DATOS CON LOS OBJETOS CREADOS

In [ ]:
training_set = train_data.flow_from_directory(
    TRAIN_DIR,
    target_size=(64,64),
    batch_size=32,
    class_mode='binary'
)
validation_set = train_data.flow_from_directory(
    VALIDATION_DIR,
    target_size=(64,64),
    batch_size=20,
    class_mode='binary'
)
test_set = train_data.flow_from_directory(
    TEST_DIR,
    target_size=(64,64),
    class_mode='binary'
)

# CREAMOS MODELO DE RED NEURONAL CONVOLUCIONAL

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout

modelo = Sequential()
modelo.add(Conv2D(32,(3,3),activation='relu',input_shape=(64,64,3)))
modelo.add(MaxPooling2D((2,2)))
modelo.add(Conv2D(32,(3,3),activation='relu'))
modelo.add(MaxPooling2D((2,2)))
modelo.add(Flatten())
modelo.add(Dense(128,activation='relu'))
modelo.add(Dropout(0.5))
modelo.add(Dense(1,activation='sigmoid'))

modelo.summary()

In [10]:
modelo.compile(loss="binary_crossentropy",
               optimizer='adam',
               metrics=['accuracy'])

# ENTRENAMOS EL MODELO

In [ ]:
history = modelo.fit(training_set,epochs=10,batch_size=100,validation_data=validation_set)

# EVALUAMOS EL MODELO

In [ ]:
evaluacion = modelo.evaluate(test_set)
print(evaluacion)

# REALIZAMOS PREDICCIÓN DEL MODELO

In [ ]:
from keras.preprocessing import image
normal = image.load_img(test_normal_dir + "/NORMAL2-IM-0035-0001.jpeg",target_size=(64,64))
pneumonia = image.load_img(test_pneumonia_dir + "/SARS-10.1148rg.242035193-g04mr34g0-Fig8a-day0.jpeg",target_size=(64,64))
imagen_prueba = pneumonia

test_image = image.img_to_array(imagen_prueba)
test_image = np.expand_dims(test_image,axis=0)
training_set.class_indices


result = modelo.predict(test_image)
print("predicción : ",result)
if result[0][0] == 0:
  print("es normal")
  plt.imshow(imagen_prueba)
  plt.title('Normal')
  plt.axis('off')
  plt.show()
else:
  print("tiene pneumonia")
  plt.imshow(imagen_prueba)
  plt.title('PNEUMONIA')
  plt.axis('off')
  plt.show()

# GRAFICAMOS ACCURACY Y LOSS

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_loss']
epochs = range(1,len(acc)+1)

plt.plot(epochs,acc,'bo',label='Training Acc')
plt.plot(epochs,val_acc,'b',label='Validation Acc')
plt.title('Precisión en validación y entrenamiento')
plt.legend()
plt.show()

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1,len(acc)+1)

plt.plot(epochs,loss,'bo',label='Training Loss')
plt.plot(epochs,val_loss,'b',label='Validation Loss')
plt.title('Pérdida en validación y entrenamiento')
plt.legend()
plt.show()